# Unlimited OCR — Colab Launcher

Clones the Fast-api-ocr project and runs it with GPU acceleration.

**Before running:** Runtime > Change runtime type > **T4 GPU**

**After running:** Open the public URL below to use the OCR dashboard with visualizations.

In [ ]:
#@title 1. Clone & Install
!git clone https://github.com/Rahid-Khan/Fast-api-ocr.git /content/Fast-api-ocr

# Install only what we need, skip Colab pre-installed packages to avoid version conflicts
!pip install -q --no-deps -r /content/Fast-api-ocr/requirements.txt
!pip install -q "Pillow>=11.0,<12.0" psutil pymupdf einops addict easydict python-multipart
!pip install -q transformers==4.57.1 2>/dev/null

# Install cloudflared (free tunnel, no account needed)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('Done')

In [ ]:
#@title 2. Run Server
import subprocess, time, threading, re

# Start uvicorn in background
proc = subprocess.Popen(
    ['uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='/content/Fast-api-ocr',
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)

# Print server logs
def log_output():
    for line in proc.stdout:
        print(line, end='')

threading.Thread(target=log_output, daemon=True).start()
time.sleep(3)

# Start cloudflared tunnel and extract public URL
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)

public_url = None
def watch_tunnel():
    global public_url
    for line in tunnel.stdout:
        print(line, end='')
        match = re.search(r'(https://[a-z0-9-]+\.trycloudflare\.com)', line)
        if match and not public_url:
            public_url = match.group(1)

threading.Thread(target=watch_tunnel, daemon=True).start()

# Wait for URL
for _ in range(30):
    if public_url:
        break
    time.sleep(1)

if public_url:
    print(f'\n{"="*50}')
    print(f'Public URL: {public_url}')
    print(f'Open this URL to use the OCR dashboard with visualizations.')
    print(f'{"="*50}')
else:
    print('\nTunnel starting... check output above for the URL.')

In [ ]:
#@title 3. Stop Server (when done)
proc.terminate()
tunnel.terminate()
print('Server stopped.')